# 01. Dataset Analysis

This notebook supports the **Dataset and Exploratory Analysis** section of the report.
It analyzes metadata, class imbalance, secondary labels, data sources, and optional audio duration statistics.

Expected files:

```text
data/train_metadata.csv
data/train_audio/
```

The notebook saves figures to `reports/figures/` and tables to `reports/tables/`.

In [ ]:
# Common setup
from pathlib import Path
import sys
import os
import json
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root: works when notebook is run from repo root or from notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    # Walk upward in case the notebook is launched from a subfolder
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            PROJECT_ROOT = parent
            break

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORT_DIR / "figures"
TABLE_DIR = REPORT_DIR / "tables"
EXP_DIR = PROJECT_ROOT / "experiments" / "notebook_outputs"
for p in [FIG_DIR, TABLE_DIR, EXP_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_DIR)

from bioacoustic.dataset import read_metadata, parse_secondary_labels, build_class_list

METADATA_PATH = DATA_DIR / "train_metadata.csv"
AUDIO_DIR = DATA_DIR / "train_audio"
PRIMARY_COL = "primary_label"
SECONDARY_COL = "secondary_labels"
FILENAME_COL = "filename"

assert METADATA_PATH.exists(), f"Missing metadata file: {METADATA_PATH}"
df = read_metadata(METADATA_PATH)
print(df.shape)
display(df.head())

## Basic dataset statistics

In [ ]:
classes = build_class_list(df, primary_col=PRIMARY_COL)
class_counts = df[PRIMARY_COL].astype(str).value_counts()

summary = {
    "num_recordings": int(len(df)),
    "num_primary_classes": int(len(classes)),
    "most_common_class": str(class_counts.index[0]),
    "most_common_count": int(class_counts.iloc[0]),
    "rarest_class": str(class_counts.index[-1]),
    "rarest_count": int(class_counts.iloc[-1]),
    "imbalance_ratio": float(class_counts.iloc[0] / max(class_counts.iloc[-1], 1)),
    "num_classes_lt_10_samples": int((class_counts < 10).sum()),
    "num_classes_gt_500_samples": int((class_counts > 500).sum()),
}

summary_df = pd.DataFrame([summary])
display(summary_df.T.rename(columns={0: "value"}))
summary_df.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
class_counts.rename_axis("class").reset_index(name="count").to_csv(TABLE_DIR / "class_distribution.csv", index=False)

## Class imbalance visualization

In [ ]:
plt.figure(figsize=(12, 4))
class_counts.head(40).plot(kind="bar")
plt.title("Top 40 most frequent primary labels")
plt.xlabel("Primary label")
plt.ylabel("Number of recordings")
plt.tight_layout()
plt.savefig(FIG_DIR / "top40_class_distribution.png", dpi=200)
plt.show()

plt.figure(figsize=(12, 4))
class_counts.tail(40).plot(kind="bar")
plt.title("Bottom 40 rarest primary labels")
plt.xlabel("Primary label")
plt.ylabel("Number of recordings")
plt.tight_layout()
plt.savefig(FIG_DIR / "bottom40_class_distribution.png", dpi=200)
plt.show()

## Data source analysis

In [ ]:
# BirdCLEF metadata may contain source-like columns with different names.
source_candidates = [c for c in df.columns if c.lower() in {"source", "data_source", "collection", "dataset", "provider"}]
print("Source-like columns:", source_candidates)

if source_candidates:
    source_col = source_candidates[0]
    source_counts = df[source_col].fillna("Unknown").astype(str).value_counts()
    display(source_counts.rename_axis("source").reset_index(name="count"))
    source_counts.rename_axis("source").reset_index(name="count").to_csv(TABLE_DIR / "source_distribution.csv", index=False)

    plt.figure(figsize=(7, 4))
    source_counts.plot(kind="bar")
    plt.title("Training samples by source")
    plt.xlabel("Source")
    plt.ylabel("Number of recordings")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "source_distribution.png", dpi=200)
    plt.show()
else:
    print("No explicit source column found. Skipping source plot.")

## Secondary-label and multi-label analysis

In [ ]:
if SECONDARY_COL in df.columns:
    secondary_lists = df[SECONDARY_COL].apply(parse_secondary_labels)
    df_secondary = df.copy()
    df_secondary["num_secondary_labels"] = secondary_lists.apply(len)
    with_secondary = int((df_secondary["num_secondary_labels"] > 0).sum())
    available = int(df[SECONDARY_COL].notna().sum())
    print(f"Rows with secondary label field available: {available}/{len(df)}")
    print(f"Rows with at least one secondary species: {with_secondary}/{len(df)}")

    sec_counts = pd.Series([x for labs in secondary_lists for x in labs]).value_counts()
    display(df_secondary["num_secondary_labels"].describe())
    df_secondary[[FILENAME_COL, PRIMARY_COL, "num_secondary_labels"]].to_csv(TABLE_DIR / "secondary_label_summary.csv", index=False)

    plt.figure(figsize=(7, 4))
    df_secondary["num_secondary_labels"].value_counts().sort_index().plot(kind="bar")
    plt.title("Number of secondary labels per recording")
    plt.xlabel("Number of secondary labels")
    plt.ylabel("Recordings")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "secondary_label_count_distribution.png", dpi=200)
    plt.show()
else:
    print("No secondary label column found.")

## Optional audio duration sampling

This cell samples a subset of audio files to estimate duration distribution. Set `MAX_DURATION_FILES=None` to scan all files, but it may be slow.

In [ ]:
MAX_DURATION_FILES = 500

def resolve_audio_path(row):
    filename = str(row[FILENAME_COL])
    direct = AUDIO_DIR / filename
    nested = AUDIO_DIR / str(row[PRIMARY_COL]) / filename
    if direct.exists():
        return direct
    if nested.exists():
        return nested
    return direct

try:
    import librosa
    sample_df = df.sample(min(MAX_DURATION_FILES or len(df), len(df)), random_state=42) if MAX_DURATION_FILES else df
    rows = []
    for _, row in sample_df.iterrows():
        path = resolve_audio_path(row)
        if path.exists():
            try:
                duration = librosa.get_duration(path=str(path))
                rows.append({"filename": row[FILENAME_COL], "primary_label": row[PRIMARY_COL], "duration_sec": duration})
            except Exception as e:
                rows.append({"filename": row[FILENAME_COL], "primary_label": row[PRIMARY_COL], "duration_sec": np.nan})
    duration_df = pd.DataFrame(rows)
    display(duration_df["duration_sec"].describe())
    duration_df.to_csv(TABLE_DIR / "audio_duration_sample.csv", index=False)

    plt.figure(figsize=(8, 4))
    duration_df["duration_sec"].dropna().clip(upper=120).hist(bins=40)
    plt.title("Audio duration distribution sample")
    plt.xlabel("Duration (seconds), clipped at 120s")
    plt.ylabel("Number of recordings")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "audio_duration_distribution.png", dpi=200)
    plt.show()
except ImportError:
    print("librosa not installed. Skipping audio duration analysis.")

## Report-ready observations

In [ ]:
print("Report notes:")
print(f"- Number of recordings: {summary['num_recordings']:,}")
print(f"- Number of primary classes: {summary['num_primary_classes']:,}")
print(f"- Most common class: {summary['most_common_class']} ({summary['most_common_count']:,} samples)")
print(f"- Rarest class: {summary['rarest_class']} ({summary['rarest_count']:,} samples)")
print(f"- Imbalance ratio: {summary['imbalance_ratio']:.1f}x")
print(f"- Classes with fewer than 10 samples: {summary['num_classes_lt_10_samples']}")